# Demo 10f — Choosing the number of clusters $k$

**Class 10 · F1** (slides: `slides/10_clustering.md`)

* The elbow of the inertia curve, found by `kneeliverse` instead of by eye.
* The silhouette coefficient by hand, and silhouette diagrams.
* Calinski-Harabasz, Davies-Bouldin, the gap statistic, and AIC/BIC for mixtures.
* A table: which $k$ each criterion picks on sets whose true $k$ is known, and the failure cases.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from kneeliverse import kneedle, lmethod
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_samples, silhouette_score
from sklearn.mixture import GaussianMixture

from faa import clustering as cl
from faa import demo, plotlib
from faa.demo import show, slide

demo.setup("10")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
COLS = [BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY]
SEED = 0
KS = np.arange(1, 11)
B = cl.benchmarks(SEED)
SETS = {
    "3 blobs": make_blobs(300, centers=3, cluster_std=0.8, random_state=SEED),
    "5 blobs": make_blobs(300, centers=5, cluster_std=0.7, random_state=3),
    "varied": B["varied"],
    "aniso": B["aniso"],
    "moons": B["moons"],
    "uniform": B["uniform"],
}
TRUE = {"3 blobs": 3, "5 blobs": 5, "varied": 3, "aniso": 3, "moons": 2, "uniform": 1}

## Slide: The elbow and its detectors

The inertia $J_k$ (within-cluster sum of squares) falls with $k$ and never reaches a minimum. The elbow is the
point after which more clusters buy little. Two detectors of `kneeliverse` read the same curve: Kneedle and the L-method.

In [ ]:
def inertia(X, ks=KS):
    return np.array([KMeans(k, n_init=10, random_state=SEED).fit(X).inertia_ for k in ks])


X3, _ = SETS["3 blobs"]
J = inertia(X3)
pts = np.c_[KS, J]
dets = {"Kneedle": kneedle.knee, "L-method": lmethod.knee}
found = {}
for name, fn in dets.items():
    try:
        found[name] = int(KS[fn(pts)])
    except Exception as e:  # a detector can refuse a short curve
        found[name] = f"error: {type(e).__name__}"
slide("The elbow and its detectors")
show(pl.DataFrame({"k": KS, "J_k": J, "drop": np.r_[np.nan, -np.diff(J)]}))
print("knee found by each detector on the 3-blob curve:", found)
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
ax.plot(KS, J, "o-", color=BLUE, ms=3)
ax.set_xlabel("$k$"), ax.set_ylabel("inertia $J_k$")
plotlib.save(f, "f1_elbow")

## Slide: The silhouette of one point

$a(i)$ = mean distance to the other points of its own cluster; $b(i)$ = mean distance to the nearest other cluster;
$s(i) = (b(i) - a(i)) / \max(a(i), b(i))$, in $[-1, 1]$.

In [ ]:
P = np.array(
    [
        [1.0, 1.0],
        [1.5, 2.0],
        [2.0, 1.5],
        [1.0, 2.5],
        [2.5, 2.0],
        [6.0, 5.0],
        [7.0, 6.0],
        [6.5, 7.0],
        [7.5, 5.5],
        [8.0, 7.0],
    ]
)
z = np.array([0] * 5 + [1] * 5)
D = np.linalg.norm(P[:, None] - P[None], axis=-1)
i = 4  # the point (2.5, 2.0)
a_i = D[i, (z == 0) & (np.arange(10) != i)].mean()
b_i = D[i, z == 1].mean()
slide("The silhouette of one point")
print(
    f"point {i} = {P[i]}: a = {a_i:.3f}, b = {b_i:.3f}, s = {(b_i - a_i) / max(a_i, b_i):.3f}; sklearn: {silhouette_samples(P, z)[i]:.3f}"
)
print(f"mean silhouette of the ten points: {silhouette_score(P, z):.3f}")
Pz = z.copy()
Pz[4] = 1  # move the border point into the wrong cluster: its silhouette turns negative
print(f"point {i} put in the wrong cluster: s = {silhouette_samples(P, Pz)[i]:.3f}")

## Slide: The silhouette diagram

One horizontal bar per point, grouped by cluster and sorted. A good $k$: all clusters wide, no negative bars,
every cluster above the mean line.

In [ ]:
f, ax = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.6), sharex=True)
rows = []
for a, k in zip(ax, (2, 3, 4, 6)):
    lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(X3)
    s = silhouette_samples(X3, lab)
    y0 = 0
    for c in range(k):
        v = np.sort(s[lab == c])
        a.fill_betweenx(np.arange(y0, y0 + len(v)), 0, v, color=COLS[c], lw=0)
        y0 += len(v) + 8
    a.axvline(s.mean(), color=RED, ls="--", lw=0.8)
    a.set_title(f"$k$ = {k}: {s.mean():.2f}"), a.set_yticks([]), a.set_xlabel("$s(i)$")
    rows.append((k, s.mean(), float(s.min()), float((s < 0).mean())))
plotlib.save(f, "f1_silhouette")
slide("The silhouette diagram")
show(pl.DataFrame(rows, schema=["k", "mean s", "min s", "share s < 0"], orient="row"))

## Slide: Five criteria, one table

For each $k = 2..8$: silhouette (max), Calinski-Harabasz (max), Davies-Bouldin (min), gap statistic (first $k$
with $\mathrm{Gap}(k) \geq \mathrm{Gap}(k+1) - s_{k+1}$), BIC and AIC of a full Gaussian mixture (min).

In [ ]:
def criteria(X, ks=range(2, 9)):
    ks = list(ks)
    sil, ch, db, bic, aic = [], [], [], [], []
    for k in ks:
        lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(X)
        (
            sil.append(silhouette_score(X, lab)),
            ch.append(calinski_harabasz_score(X, lab)),
            db.append(davies_bouldin_score(X, lab)),
        )
        g = GaussianMixture(k, n_init=3, random_state=SEED).fit(X)
        bic.append(g.bic(X)), aic.append(g.aic(X))
    gap, se = cl.gap_statistic(X, [1, *ks], seed=SEED)
    k_gap = next((k for k, g0, g1, s1 in zip([1, *ks], gap, gap[1:], se[1:]) if g0 >= g1 - s1), ks[-1])
    return {
        "silhouette": ks[int(np.argmax(sil))],
        "CH": ks[int(np.argmax(ch))],
        "DB": ks[int(np.argmin(db))],
        "gap": k_gap,
        "BIC": ks[int(np.argmin(bic))],
        "AIC": ks[int(np.argmin(aic))],
    }, (ks, sil, ch, db, bic, aic, gap, se)


picked, curves = {}, {}
for name, (X, _) in SETS.items():
    picked[name], curves[name] = criteria(X)
slide("Five criteria, one table")
show(
    pl.DataFrame(
        [(n, TRUE[n], *picked[n].values()) for n in SETS],
        schema=["set", "true k", *picked["3 blobs"].keys()],
        orient="row",
    )
)
ok = {c: sum(picked[n][c] == TRUE[n] for n in SETS) for c in picked["3 blobs"]}
print("sets where the criterion picks the true k (of 6):", ok)

## Slide: BIC and AIC for mixtures

$\mathrm{AIC} = 2p - 2\ln\hat{L}$, $\mathrm{BIC} = p\ln n - 2\ln\hat{L}$, with $p$ free parameters. BIC penalises
harder once $n > 7$, so it picks fewer components. On the 5-blob set the curves have the same knee.

In [ ]:
ks, sil, ch, db, bic, aic, gap, se = curves["5 blobs"]
f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.45))
ax[0].plot(ks, sil, "o-", color=BLUE, ms=3), ax[0].set_title("silhouette"), ax[0].set_xlabel("$k$")
(
    ax[1].plot(ks, np.array(bic) - min(bic), "o-", color=ORANGE, ms=3, label="BIC"),
    ax[1].plot(ks, np.array(aic) - min(aic), "s-", color=GREEN, ms=3, label="AIC"),
)
(
    ax[1].set_title("GMM, minus its minimum"),
    ax[1].set_xlabel("components"),
    ax[1].set_yscale("symlog", linthresh=10),
    ax[1].legend(),
)
(
    ax[2].errorbar([1, *ks], gap, yerr=se, fmt="o-", color=RED, ms=3, lw=0.8),
    ax[2].set_title("gap statistic"),
    ax[2].set_xlabel("$k$"),
)
plotlib.save(f, "f1_criteria")
slide("BIC and AIC for mixtures")
print(
    f"5 blobs: silhouette {ks[int(np.argmax(sil))]}, BIC {ks[int(np.argmin(bic))]}, AIC {ks[int(np.argmin(aic))]}; CH {ks[int(np.argmax(ch))]}"
)
n = 300
print(f"BIC penalty per parameter ln(n) = {np.log(n):.2f} against 2 for AIC")

## Failure: structure that is not there, structure that is not round

On uniform data every criterion still returns a $k$, and Kneedle finds a knee in a smooth curve; only the gap
statistic (against a uniform reference) may say $k = 1$. On the moons every $k$-means based criterion keeps asking for more
clusters: choosing $k$ does not repair the model.

In [ ]:
Xu, _ = SETS["uniform"]
Ju = inertia(Xu)
ku = int(KS[kneedle.knee(np.c_[KS, Ju])])
slide("Structure that is not there")
print(
    f"uniform data: Kneedle knee at k = {ku}; silhouette at k = 3: {silhouette_score(Xu, KMeans(3, n_init=10, random_state=SEED).fit_predict(Xu)):.3f}; "
    f"3 real blobs at k = 3: {silhouette_score(X3, KMeans(3, n_init=10, random_state=SEED).fit_predict(X3)):.3f}"
)
print("criteria on uniform data:", picked["uniform"])
print("criteria on the moons:", picked["moons"])

## Sweep: how stable is the choice?

Twenty fresh samples of the 5-blob and 3-blob sets: the share of runs in which each criterion finds the true $k$.

In [ ]:
rows = []
for name, centers, std in (("3 blobs", 3, 1.2), ("5 blobs", 5, 1.0)):
    hits = {c: 0 for c in picked["3 blobs"]}
    for s in range(20):
        X, _ = make_blobs(300, centers=centers, cluster_std=std, random_state=100 + s)
        p, _ = criteria(X)
        for c in hits:
            hits[c] += p[c] == centers
    rows.append((f"{name}, std {std}", *[hits[c] / 20 for c in hits]))
slide("Stability of the choice")
show(pl.DataFrame(rows, schema=["set", *picked["3 blobs"].keys()], orient="row"))

## Slide: Two detectors on one curve (figure)

The inertia curve of three blobs with the knee of each `kneeliverse` detector marked.

In [ ]:
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
ax.plot(KS, J, "-", color=BLUE, lw=1)
for (name, kk), c in zip(found.items(), (RED, GREEN)):
    ax.scatter([kk], [J[kk - 1]], color=c, s=26, label=f"{name}: {kk}", zorder=3)
ax.set_xlabel("$k$"), ax.set_ylabel("inertia"), ax.legend(fontsize=5.8)
plotlib.save(f, "f1_detectors")

## Slide: Calinski-Harabasz and Davies-Bouldin (figure)

Both indices for $k = 2..8$ on the 3-blob set: CH peaks and DB dips at the same $k$.

In [ ]:
ks3, sil3, ch3, db3, bic3, aic3, gap3, se3 = curves["3 blobs"]
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 0.62))
ax[0].plot(ks3, ch3, "o-", color=BLUE, ms=3), ax[0].set_title("Calinski-Harabasz (max)"), ax[0].set_xlabel("$k$")
ax[1].plot(ks3, db3, "o-", color=ORANGE, ms=3), ax[1].set_title("Davies-Bouldin (min)"), ax[1].set_xlabel("$k$")
plotlib.save(f, "f1_ch_db")
slide("Calinski-Harabasz and Davies-Bouldin")
print(f"3 blobs: CH {[round(v) for v in ch3]}, DB {[round(v, 2) for v in db3]}")

## Slide: Which criterion, on which set (figure)

Share of 20 fresh draws in which each criterion finds the true $k$ (the table above), as bars.

In [ ]:
names_c = list(picked["3 blobs"].keys())
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.4))
for q, (row, c) in enumerate(zip(rows, (BLUE, ORANGE))):
    ax.bar(np.arange(len(names_c)) + (q - 0.5) * 0.38, row[1:], 0.38, color=c, label=row[0])
ax.set_xticks(range(len(names_c)), names_c), ax.set_ylabel("share correct"), ax.legend(fontsize=6)
plotlib.save(f, "f1_hits")

## Figures: the gap statistic and the sets without structure

Gap curves of three blobs and uniform data; silhouette curves of the moons, uniform data and three blobs.

In [ ]:
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
for name, c in (("3 blobs", BLUE), ("uniform", RED)):
    ks_, sil_, ch_, db_, bic_, aic_, gap_, se_ = curves[name]
    ax.errorbar([1, *ks_], gap_, yerr=se_, fmt="o-", ms=3, lw=0.8, color=c, label=name)
ax.set_xlabel("$k$"), ax.set_ylabel("gap"), ax.legend()
plotlib.save(f, "f1_gap")
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
for name, c, mk in (("3 blobs", BLUE, "o-"), ("uniform", RED, "o-"), ("moons", GREEN, "s--")):
    ax.plot(curves[name][0], curves[name][1], mk, ms=3, color=c, label=name)
ax.set_xlabel("$k$"), ax.set_ylabel("silhouette"), ax.legend()
plotlib.save(f, "f1_nostructure")
slide("The gap and the silhouette without structure")
print("silhouette at k = 3:", {n: round(float(curves[n][1][1]), 3) for n in ("3 blobs", "uniform", "moons")})

## Slide: How the two detectors work

**Kneedle**: normalise the curve to the unit square and take the point farthest above the diagonal (the difference curve).
**L-method**: fit two straight lines, one left and one right of each split, and take the split with the smallest weighted
RMSE. Both read the same inertia curve of three blobs.

In [ ]:
ks = KS.astype(float)
pts = np.c_[KS, J]
xn, yn = (ks - ks.min()) / (ks.max() - ks.min()), (J - J.min()) / (J.max() - J.min())
diff = (1 - yn) - xn  # Kneedle difference curve of a decreasing convex curve
split = range(2, 9)


def rmse(x, y):
    A = np.c_[x, np.ones_like(x)]
    c = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(np.sqrt(((A @ c - y) ** 2).mean()))


lm = [(c / 10) * rmse(ks[:c], J[:c]) + ((11 - c) / 10) * rmse(ks[c - 1 :], J[c - 1 :]) for c in split]
slide("How the two detectors work")
show(pl.DataFrame({"k": KS, "J_k": J, "Kneedle difference": diff}))
print(f"Kneedle difference peaks at k = {KS[int(np.argmax(diff))]} ({diff.max():.3f})")
print(
    f"L-method: smallest weighted RMSE at k = {list(split)[int(np.argmin(lm))]} ({min(lm):.1f}); at k = 2: {lm[0]:.1f}, k = 4: {lm[2]:.1f}"
)
print(
    "library calls on the raw curve:",
    {n: int(KS[f(pts)]) for n, f in {"Kneedle": kneedle.knee, "L-method": lmethod.knee}.items()},
)

f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.25))
ax[0].plot(KS, yn, "o-", color=BLUE, ms=3, label="curve")
ax[0].plot(KS, 1 - xn, "--", color=GREY, lw=0.8, label="diagonal")
ax[0].plot(KS, diff, "s-", color=ORANGE, ms=3, label="difference")
ax[0].set_title("Kneedle", fontsize=7)
ax[0].legend(fontsize=5.5)
ax[1].plot(KS, J, "o", color=BLUE, ms=3)
for sl in (slice(0, 3), slice(2, None)):
    fit = np.polyfit(ks[sl], J[sl], 1)
    ax[1].plot(ks[sl], np.polyval(fit, ks[sl]), color=GREEN, lw=1.2)
ax[1].set_title("L-method, split $k$ = 3", fontsize=7)
ax[2].plot(list(split), lm, "o-", color=GREEN, ms=3)
ax[2].set_title("L-method error", fontsize=7)
for a in ax:
    a.set_xlabel("$k$")
plotlib.save(f, "f1_knee_geometry")